# Combining Datasets with Pandas

This notebook teaches the main ways to combine datasets using **3 files from the Datos Wansoft dataset**.

Each file represents sales transactions from a different Panem restaurant branch:
- `Carreta` — Feb–May 2022
- `Punto Valle` — Feb–May 2022
- `Plaza QIN` — Apr–May 2022

---

## Methods covered

| Method | Use case |
|---|---|
| `pd.concat()` axis=0 | Stack rows (same columns, different data) |
| `pd.concat()` axis=1 | Stack columns side by side |
| `glob` + loop | Load and combine many files at once |

---
## Example: Load the 3 files

In [ ]:
import pandas as pd
import glob

DATA_DIR =  

# --- Load the 3 files individually ---
df_carreta = pd.read_csv(DATA_DIR + "detail_Panem-Carreta_2022-02-18_2022-05-07.csv")
df_punto   = pd.read_csv(DATA_DIR + "detail_Panem-Punto-Valle_2022-02-08_2022-05-08.csv")
df_qin     = pd.read_csv(DATA_DIR + "detail_Panem-Plaza-QIN_2022-04-28_2022-05-08.csv")

print("Shapes:")
print(f"  Carreta:     {df_carreta.shape}")
print(f"  Punto Valle: {df_punto.shape}")
print(f"  Plaza QIN:   {df_qin.shape}")

# Quick look at one of them
df_carreta.head(3)

Shapes:
  Carreta:     (21524, 46)
  Punto Valle: (51068, 46)
  Plaza QIN:   (5067, 46)


,sucursal,operating_date,day_name,closing_time,captured_time,week_number,pdv_txn_id,order_id,order_type,order_subtype,...,total_item,subtotal_cortesia_cancel,iva_cortesia_cancel,ieps_cortesia_cancel,total_cortesia_cancel,subtotal_anulacion,iva_anulacion,ieps_anulacion,total_anulacion,clave_platillo
0,Panem - Carreta,2022-02-18,viernes,2022-02-18 08:21:04.887000,2022-02-18 08:17:48.077000,7,1,1,Para llevar,'-,...,201.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,DE005
1,Panem - Carreta,2022-02-18,viernes,2022-02-18 08:21:04.887000,2022-02-18 08:17:52.763000,7,1,1,Para llevar,'-,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MA001
2,Panem - Carreta,2022-02-18,viernes,2022-02-18 08:21:04.887000,2022-02-18 08:17:53.950000,7,1,1,Para llevar,'-,...,12.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MA003


---
## `pd.concat()` axis=0: Stack rows vertically

**When to use:** All files have the **same columns** and you want to build one unified table.

This is the most common operation when files represent different time periods or locations with the same schema.

In [2]:
# --- Basic concat: just stack the rows ---
df_all = pd.concat([df_carreta, df_punto, df_qin])

print(f"Individual sizes: {len(df_carreta)}, {len(df_punto)}, {len(df_qin)}")
print(f"Combined size:    {len(df_all)}  (should equal the sum above)")

# Notice the index is duplicated — rows from each file keep their original index
print("\nFirst few index values:", df_all.index[:6].tolist())
df_all.head(3)

Individual sizes: 21524, 51068, 5067
Combined size:    77659  (should equal the sum above)

First few index values: [0, 1, 2, 3, 4, 5]


,sucursal,operating_date,day_name,closing_time,captured_time,week_number,pdv_txn_id,order_id,order_type,order_subtype,...,total_item,subtotal_cortesia_cancel,iva_cortesia_cancel,ieps_cortesia_cancel,total_cortesia_cancel,subtotal_anulacion,iva_anulacion,ieps_anulacion,total_anulacion,clave_platillo
0,Panem - Carreta,2022-02-18,viernes,2022-02-18 08:21:04.887000,2022-02-18 08:17:48.077000,7,1,1,Para llevar,'-,...,201.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,DE005
1,Panem - Carreta,2022-02-18,viernes,2022-02-18 08:21:04.887000,2022-02-18 08:17:52.763000,7,1,1,Para llevar,'-,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MA001
2,Panem - Carreta,2022-02-18,viernes,2022-02-18 08:21:04.887000,2022-02-18 08:17:53.950000,7,1,1,Para llevar,'-,...,12.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MA003


In [3]:
# --- ignore_index=True resets the index so it is clean and continuous ---
df_all = pd.concat([df_carreta, df_punto, df_qin], ignore_index=True)

print("Index after ignore_index=True:", df_all.index[:6].tolist())
print(f"Shape: {df_all.shape}")

Index after ignore_index=True: [0, 1, 2, 3, 4, 5]
Shape: (77659, 46)


---
## `glob` + loop: Combine many files at once

**When to use:** You have dozens of files with the same schema and don't want to load them one by one.

This is the real-world pattern for this dataset (150+ files).

In [12]:
# --- Load every Carreta file (all periods) with one pattern ---
carreta_files = glob.glob(DATA_DIR + "detail_Panem-Carreta_*.csv")
print(f"Carreta files found: {len(carreta_files)}")
for f in sorted(carreta_files):
    print(" ", f.split("/")[-1])

Carreta files found: 7
  detail_Panem-Carreta_2022-02-18_2022-05-07.csv
  detail_Panem-Carreta_2022-05-09_2022-08-05.csv
  detail_Panem-Carreta_2022-08-08_2022-11-04.csv
  detail_Panem-Carreta_2022-11-05_2023-02-02.csv
  detail_Panem-Carreta_2023-02-03_2023-05-03.csv
  detail_Panem-Carreta_2023-05-04_2023-08-01.csv
  detail_Panem-Carreta_2023-08-02_2023-10-30.csv


In [13]:
# --- Combine all of them into one DataFrame ---
df_carreta_full = pd.concat(
    [pd.read_csv(f) for f in sorted(carreta_files)],
    ignore_index=True
)

print(f"Total Carreta rows (all periods): {len(df_carreta_full):,}")
print(f"Date range: {df_carreta_full['operating_date'].min()}  →  {df_carreta_full['operating_date'].max()}")

/var/folders/1d/2nnpvlqj4wg78b81f6hxfw300000gp/T/ipykernel_75862/4152468081.py:3: DtypeWarning: Columns (21) have mixed types. Specify dtype option on import or set low_memory=False.
  [pd.read_csv(f) for f in sorted(carreta_files)],


Total Carreta rows (all periods): 171,360
Date range: 2022-02-18  →  2023-10-30


In [14]:
# --- Or load ALL files from ALL branches at once ---
all_files = glob.glob(DATA_DIR + "detail_*.csv")
print(f"Total files: {len(all_files)}")

df_complete = pd.concat(
    [pd.read_csv(f) for f in all_files],
    ignore_index=True
)

print(f"Total rows: {len(df_complete):,}")
print("\nRows per branch:")
print(df_complete["sucursal"].value_counts())

Total files: 186


/var/folders/1d/2nnpvlqj4wg78b81f6hxfw300000gp/T/ipykernel_75862/2627673863.py:6: DtypeWarning: Columns (21) have mixed types. Specify dtype option on import or set low_memory=False.
  [pd.read_csv(f) for f in all_files],
/var/folders/1d/2nnpvlqj4wg78b81f6hxfw300000gp/T/ipykernel_75862/2627673863.py:6: DtypeWarning: Columns (21) have mixed types. Specify dtype option on import or set low_memory=False.
  [pd.read_csv(f) for f in all_files],
/var/folders/1d/2nnpvlqj4wg78b81f6hxfw300000gp/T/ipykernel_75862/2627673863.py:6: DtypeWarning: Columns (21) have mixed types. Specify dtype option on import or set low_memory=False.
  [pd.read_csv(f) for f in all_files],
/var/folders/1d/2nnpvlqj4wg78b81f6hxfw300000gp/T/ipykernel_75862/2627673863.py:6: DtypeWarning: Columns (21) have mixed types. Specify dtype option on import or set low_memory=False.
  [pd.read_csv(f) for f in all_files],
/var/folders/1d/2nnpvlqj4wg78b81f6hxfw300000gp/T/ipykernel_75862/2627673863.py:6: DtypeWarning: Columns (21) hav

Total rows: 3,764,683

Rows per branch:
sucursal
Panem - Punto Valle            944346
Panem - Hotel Kavia N          602608
Panem - Plaza QIN N            385516
Panem - Plaza QIN              349364
Panem - Hospital Zambrano N    335642
Panem - Hotel Kavia            254154
Panem - Hospital Zambrano      251304
Panem - La Carreta N           227499
Panem - Plaza Nativa           194511
Panem - Carreta                171360
Panem - Credi Club              48379
Name: count, dtype: int64
